In [9]:
import os
import h5py
import numpy as np
import pandas as pd
import scipy as sp
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, resample_poly
from scipy.signal import butter, filtfilt, resample_poly, resample, find_peaks
import neurokit2 as nk
from scipy.signal import find_peaks
import seaborn as sns
from scipy.stats import wilcoxon
import scipy

# Pandas display settings (optional)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)

In [12]:
# ── paths ──────────────────────────────────────────────────────────────
h5_dir   = r"C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\BLA_ChR_resp_pilot1 (SS)\BLA_resp_ChR_cm_hc\h5_outputs"
boris_dir = r"C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\BLA_ChR_resp_pilot1 (SS)\BLA_resp_ChR_cm_boris"
# ── pick one file of each to inspect ───────────────────────────────────
h5_files    = [f for f in os.listdir(h5_dir)    if f.endswith('.h5')]
boris_files = [f for f in os.listdir(boris_dir) if f.endswith('.csv')]

print("H5 files found:")
for i, f in enumerate(h5_files):
    print(f"  [{i}] {f}")

print("\nBORIS files found:")
for i, f in enumerate(boris_files):
    print(f"  [{i}] {f}")

# ── inspect the first H5 ────────────────────────────────────────────────
h5_path = os.path.join(h5_dir, h5_files[0])
print(f"\n--- Inspecting H5: {h5_files[0]} ---")

with h5py.File(h5_path, 'r') as f:
    print("Root attrs:", dict(f.attrs))
    def show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  DATASET '{name}': shape={obj.shape}, dtype={obj.dtype}, attrs={dict(obj.attrs)}")
            if obj.shape[0] < 20:
                print(f"    values: {obj[:]}")
            else:
                print(f"    first 5: {obj[:5]}")
                print(f"    last  5: {obj[-5:]}")
        elif isinstance(obj, h5py.Group):
            print(f"  GROUP   '{name}': attrs={dict(obj.attrs)}")
    f.visititems(show)

# ── inspect the first BORIS CSV ─────────────────────────────────────────
boris_path = os.path.join(boris_dir, boris_files[0])
print(f"\n--- Inspecting BORIS: {boris_files[0]} ---")

df = pd.read_csv(boris_path)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df[['Subject','Behavior','Start (s)','Stop (s)','Image index start','Image index stop']].head(10).to_string())

H5 files found:
  [0] 1_1_i_cm_1_2_d_20260106_134350.h5
  [1] 1_1_i_cm_1_3_s_20260107_163611.h5
  [2] 1_2_d_cm_1_3_s_20260108_114416.h5
  [3] 1_3_s_1_2_d_20260107_165340.h5
  [4] 1_3_s_cm_1_1_i_20260106_122823.h5
  [5] 2_1_d_cm_2.3_i_20260106_105109.h5
  [6] 2_1_d_cm_2_2_s_20260108_112426.h5
  [7] 2_2_s_cm_2_1_d_20260108_120453.h5
  [8] 2_2_s_cm_2_1_d_20260108_120453.rec.h5
  [9] 2_2_s_cm_2_3_i_20260106_132549.h5
  [10] 3_1_i_cm_3_2_d_20260106_115157.h5
  [11] 3_1_i_cm_3_3_s_20260107_123230.h5
  [12] 3_2_d_cm_3_3_s_20260107_140924.h5
  [13] 3_3_s_cm_3_1_i_20260106_124546.h5
  [14] 3_3_s_cm_3_2_d_20260107_171330.h5
  [15] 4_1_s_cm_4_2_d_20260107_160439.h5
  [16] 4_2_d_cm_4_1_s_20260107_150836.h5
  [17] 4_3_d_cm_4_1_s_20260107_113120.h5
  [18] 5_1_i_cm_5_2_d_20260106_120946.h5
  [19] 5_1_i_cm_5_3_s_20260107_132309.h5
  [20] 6_2_d_cm_6_3_s_20260107_154939.h5
  [21] 7_1_d_cm_7_2_s_20260106_111330.h5
  [22] 7_2_s_cm_7_1_d_20260106_130431.h5
  [23] 8_1_s_cm_8_2_d_20260107_152812.h5
  [24] 8_

In [14]:
print(df.columns.tolist())

['Observation id', 'Observation date', 'Description', 'Observation type', 'Source', 'Time offset (s)', 'Coding duration', 'Media duration (s)', 'FPS (frame/s)', 'Subject', 'Observation duration by subject by observation', 'Behavior', 'Behavioral category', 'Behavior type', 'Start (s)', 'Stop (s)', 'Duration (s)', 'Media file name', 'Image index start', 'Image index stop', 'Image file path start', 'Image file path stop', 'Comment start', 'Comment stop']


In [15]:
print(df[['Subject','Behavior','Start (s)','Stop (s)','Image index start','Image index stop']].head(10).to_string())

        Subject             Behavior  Start (s)  Stop (s)  Image index start  Image index stop
0  social_agent        body sniffing     24.867    26.533              373.0             398.0
1  social_agent        body sniffing     28.600    29.667              429.0             445.0
2  social_agent        body sniffing     30.933    31.667              464.0             475.0
3  social_agent      facial sniffing     40.933    42.133              614.0             632.0
4       subject      facial sniffing     41.067    42.333              616.0             635.0
5       subject      facial sniffing     44.933    48.467              674.0             727.0
6  social_agent      facial sniffing     44.933    48.267              674.0             724.0
7  social_agent  anogenital sniffing     50.533    51.733              758.0             776.0
8       subject      facial sniffing     59.800    60.733              897.0             911.0
9  social_agent              digging     77.867   

In [16]:
print(df["Subject"].unique())

print(df["Behavior"].unique())

['social_agent' 'subject']
['body sniffing' 'facial sniffing' 'anogenital sniffing' 'digging'
 'self-grooming']


In [23]:
# ======================================================
# PROCESS ONE BORIS FILE
# FULL METADATA VERSION
# ======================================================

import re

beh_df = df.copy()

# ------------------------------------------------------
# parse filename
# ------------------------------------------------------

filename = boris_files[0]

print(filename)

match = re.search(
    r'(\d+)_(\d+)_2_([dis])_cm_(\d+)_(\d+)_([dis])',
    filename
)

rank_map = {
    "d": "DOM",
    "i": "INT",
    "s": "SUB"
}

# ------------------------------------------------------
# parse filename
# ------------------------------------------------------

match = re.search(
    r'(\d+)_(\d+)_(\d+)_([dis])_cm_(\d+)_(\d+)_([dis])',
    filename
)

# ------------------------------------------------------
# IDs
# ------------------------------------------------------

subject_id = (
    f"{match.group(1)}_{match.group(2)}"
)

social_agent_id = (
    f"{match.group(5)}_{match.group(6)}"
)

# ------------------------------------------------------
# cohort
# ------------------------------------------------------

cohort = match.group(3)

# ------------------------------------------------------
# ranks
# ------------------------------------------------------

subject_rank = rank_map[
    match.group(4)
]

social_agent_rank = rank_map[
    match.group(7)
]

# ------------------------------------------------------
# keep only social sniffing
# ------------------------------------------------------

social_behaviors = [
    "facial sniffing",
    "body sniffing",
    "anogenital sniffing"
]

beh_df = beh_df[
    beh_df["Behavior"].isin(
        social_behaviors
    )
].copy()

# ------------------------------------------------------
# duration
# ------------------------------------------------------

beh_df["InteractionDuration"] = (
    beh_df["Stop (s)"]
    -
    beh_df["Start (s)"]
)

# ------------------------------------------------------
# metadata columns
# ------------------------------------------------------

beh_df["SubjectID"] = subject_id

beh_df["SocialAgentID"] = (
    social_agent_id
)

beh_df["Cohort"] = cohort

beh_df["SubjectRank"] = (
    subject_rank
)

beh_df["SocialAgentRank"] = (
    social_agent_rank
)

# ------------------------------------------------------
# subject-relative interaction label
# ------------------------------------------------------

rank_order = {
    "SUB": 0,
    "INT": 1,
    "DOM": 2
}

if rank_order[subject_rank] > rank_order[social_agent_rank]:

    relative_label = "DOM"

elif rank_order[subject_rank] < rank_order[social_agent_rank]:

    relative_label = "SUB"

else:

    relative_label = "EQUAL"

beh_df["RelativeInteraction"] = (
    relative_label
)

# ------------------------------------------------------
# rename actor column
# ------------------------------------------------------

beh_df = beh_df.rename(
    columns={
        "Subject": "Actor"
    }
)

# ------------------------------------------------------
# preview
# ------------------------------------------------------

display(

    beh_df[[
        "Actor",
        "Behavior",
        "InteractionDuration",
        "SubjectID",
        "SocialAgentID",
        "Cohort",
        "SubjectRank",
        "SocialAgentRank",
        "RelativeInteraction"
    ]].head(20)

)

1_1_2_i_cm_1_2_d_20260106_134350.1_BORIS_SS.csv


,Actor,Behavior,InteractionDuration,SubjectID,SocialAgentID,Cohort,SubjectRank,SocialAgentRank,RelativeInteraction
0,social_agent,body sniffing,1.666,1_1,1_2,2,INT,DOM,SUB
1,social_agent,body sniffing,1.067,1_1,1_2,2,INT,DOM,SUB
2,social_agent,body sniffing,0.734,1_1,1_2,2,INT,DOM,SUB
3,social_agent,facial sniffing,1.200,1_1,1_2,2,INT,DOM,SUB
4,subject,facial sniffing,1.266,1_1,1_2,2,INT,DOM,SUB
5,subject,facial sniffing,3.534,1_1,1_2,2,INT,DOM,SUB
6,social_agent,facial sniffing,3.334,1_1,1_2,2,INT,DOM,SUB
7,social_agent,anogenital sniffing,1.200,1_1,1_2,2,INT,DOM,SUB
8,subject,facial sniffing,0.933,1_1,1_2,2,INT,DOM,SUB
11,subject,facial sniffing,0.733,1_1,1_2,2,INT,DOM,SUB


In [24]:
# ======================================================
# COHORT 1 PARSER
# ======================================================

import glob
import os
import re
import pandas as pd

cohort1_dir = (
    r"C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_h5\baseline_cagemate_interactions_h5_outputs"
)

cohort1_files = glob.glob(
    os.path.join(cohort1_dir, "*.h5")
)

rows = []

for path in cohort1_files:

    fname = os.path.basename(path)

    # remove extension
    stem = fname.replace(".h5", "")

    # Example:
    # CM_s1_1_d1_2_20250623_111352_merged

    # or:
    # CM_s1_2_sub1_1_20250623_133932_merged

    match = re.search(
        r'CM_s(\d+)_(\d+)_(d|sub)(\d+)_(\d+)',
        stem
    )

    if match is None:

        print("FAILED:", fname)
        continue

    # --------------------------------------------------
    # IDs
    # --------------------------------------------------

    subject_id = (
        f"{match.group(1)}_{match.group(2)}"
    )

    social_agent_id = (
        f"{match.group(4)}_{match.group(5)}"
    )

    # --------------------------------------------------
    # rank
    # --------------------------------------------------

    relation = match.group(3)

    if relation == "d":

        subject_rank = "DOM"
        social_agent_rank = "SUB"

    elif relation == "sub":

        subject_rank = "SUB"
        social_agent_rank = "DOM"

    else:

        subject_rank = "UNKNOWN"
        social_agent_rank = "UNKNOWN"

    # --------------------------------------------------
    # relative interaction
    # --------------------------------------------------

    relative_interaction = (
        subject_rank
        + "-" +
        social_agent_rank
    )

    # --------------------------------------------------
    # row
    # --------------------------------------------------

    row = {

        "Cohort": "1",

        "SubjectID":
            subject_id,

        "SocialAgentID":
            social_agent_id,

        "SubjectRank":
            subject_rank,

        "SocialAgentRank":
            social_agent_rank,

        "RelativeInteraction":
            relative_interaction,

        "H5Path":
            path,

        "Filename":
            fname
    }

    rows.append(row)

# ======================================================
# DATAFRAME
# ======================================================

cohort1_meta = pd.DataFrame(rows)

print(cohort1_meta.shape)

display(cohort1_meta)

(9, 8)


,Cohort,SubjectID,SocialAgentID,SubjectRank,SocialAgentRank,RelativeInteraction,H5Path,Filename
0,1,1_1,1_2,DOM,SUB,DOM-SUB,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_h5\baseline_cagemate_interactions_h5_outputs\CM_s1_1_d1_2_20250623_111352_merged.h5,CM_s1_1_d1_2_20250623_111352_merged.h5
1,1,1_2,1_1,SUB,DOM,SUB-DOM,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_h5\baseline_cagemate_interactions_h5_outputs\CM_s1_2_sub1_1_20250623_133932_merged.h5,CM_s1_2_sub1_1_20250623_133932_merged.h5
2,1,2_3,2_4,DOM,SUB,DOM-SUB,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_h5\baseline_cagemate_interactions_h5_outputs\CM_s2_3_d2_4_20250623_151153_merged.h5,CM_s2_3_d2_4_20250623_151153_merged.h5
3,1,2_4,2_3,SUB,DOM,SUB-DOM,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_h5\baseline_cagemate_interactions_h5_outputs\CM_s2_4_sub2_3_20250623_143348_merged.h5,CM_s2_4_sub2_3_20250623_143348_merged.h5
4,1,3_5,3_6,DOM,SUB,DOM-SUB,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_h5\baseline_cagemate_interactions_h5_outputs\CM_s3_5_d3_6_20250623_160001_merged.h5,CM_s3_5_d3_6_20250623_160001_merged.h5
5,1,3_5,3_6,DOM,SUB,DOM-SUB,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_h5\baseline_cagemate_interactions_h5_outputs\CM_s3_5_d3_6_20250623_170708_merged.h5,CM_s3_5_d3_6_20250623_170708_merged.h5
6,1,3_6,3_5,SUB,DOM,SUB-DOM,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_h5\baseline_cagemate_interactions_h5_outputs\CM_s3_6_sub3_5_20250623_174348_merged.h5,CM_s3_6_sub3_5_20250623_174348_merged.h5
7,1,4_7,4_8,DOM,SUB,DOM-SUB,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_h5\baseline_cagemate_interactions_h5_outputs\CM_s4_7_d4_8_20250623_193718_merged.h5,CM_s4_7_d4_8_20250623_193718_merged.h5
8,1,4_8,4_7,SUB,DOM,SUB-DOM,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_h5\baseline_cagemate_interactions_h5_outputs\CM_s4_8_sub4_7_20250623_182649_merged.h5,CM_s4_8_sub4_7_20250623_182649_merged.h5
